# Phase 4C.2 — Stage 2 Fine-Tuning Launcher

- **Mục tiêu**: Huấn luyện 15 runs Stage 2 partial fine-tuning ($N \in \{50, 100, 250\} \times 5$ seeds).
- **Runtime Target**: Capability-based Python runtime (yêu cầu Python $\ge 3.10$, kiểm thử thực tế trên Python 3.13.15) kèm PyTorch và NVIDIA CUDA GPU (Tesla T4, L4, V100, A100 hoặc tương đương với $\ge 8.0$ GB VRAM).
- **Dataset**: Tái sử dụng `phase_4c1_binary_n250_reusable.tar` từ `phase_4c1/inputs/` ở chế độ read-only.
- **Code & Operator**: Đặt tại `phase_4c2/inputs/` trên Google Drive.
- **Chế độ mặc định**: `EXECUTE = False` (chạy kiểm tra tiền kiểm `--preflight-only`, không huấn luyện).
- **Thực thi**: Đổi `EXECUTE = True` và chọn **Run all** để chạy 15 runs có khả năng resume tự động.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

EXECUTE = False

# Google Drive Paths
DRIVE_PHASE4C1_DIR = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c1")
DRIVE_DATASET_PATH = DRIVE_PHASE4C1_DIR / "inputs" / "phase_4c1_binary_n250_reusable.tar"

DRIVE_PHASE4C2_DIR = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c2")
DRIVE_INPUT_DIR = DRIVE_PHASE4C2_DIR / "inputs"
DRIVE_OUTPUT_DIR = DRIVE_PHASE4C2_DIR / "runs"

LOCAL_TRANSFER_DIR = Path("/content/forensics-stage2-transfer")
LOCAL_OUTPUT_PATH = Path("/content/phase_4c2_outputs")

CANONICAL_DATASET_SHA = "d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27"
CANONICAL_DATASET_BYTES = 724633600

CANONICAL_CODE_ARCHIVE_NAME = "phase_4c2_code_9ee7fdb.tar.gz"
CANONICAL_CODE_ARCHIVE_BYTES = 10478136
CANONICAL_CODE_ARCHIVE_SHA256 = "951e9089582eb60cf3d293c37982a8f3c3b6a3e05fb3ef45f23c666d41bc7d89"
CANONICAL_EXECUTION_SHORT_SHA = "9ee7fdb"

CANONICAL_OPERATOR_SHA = "2a967a475c9bdc45515addc7123b8312f2d180b5e21fc735d8e6f7f5f4aa8929"
CANONICAL_OPERATOR_BYTES = 64776

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
drive.mount("/content/drive", force_remount=False)

assert sys.platform.startswith("linux"), f"Yêu cầu Linux, nhận được: {sys.platform}"

# Capability-based Python runtime preflight check
py_version_str = ".".join(map(str, sys.version_info[:3]))
assert sys.version_info >= (3, 10), f"Yêu cầu Python >= 3.10, nhận được: {py_version_str}"
print(f"[+] Python runtime: {py_version_str} (capability check PASS)")

if EXECUTE:
    assert torch.cuda.is_available(), "CUDA không khả dụng khi EXECUTE=True"
    gpu_model = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    assert vram_gb >= 8.0, f"VRAM không đủ: {vram_gb:.2f} GB (yêu cầu >= 8.0 GB)"
    x = torch.ones((32, 32), device="cuda")
    y = x @ x
    torch.cuda.synchronize()
    assert y.is_cuda and torch.isfinite(y).all(), "CUDA smoke test failed"
    print(f"[+] GPU phát hiện: {gpu_model} ({vram_gb:.2f} GB VRAM) — Capability PASS")

# Preflight: fail-closed if inputs directory missing (DO NOT mkdir on inputs)
if not DRIVE_INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục inputs đã chuẩn bị trên Drive: {DRIVE_INPUT_DIR}")

if not DRIVE_DATASET_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy dataset Stage 1 tại: {DRIVE_DATASET_PATH}")

# Verify dataset size and SHA-256 (streaming by chunk)
assert DRIVE_DATASET_PATH.stat().st_size == CANONICAL_DATASET_BYTES, f"Sai kích thước dataset: {DRIVE_DATASET_PATH}"
actual_ds_sha = sha256_file(DRIVE_DATASET_PATH)
assert actual_ds_sha == CANONICAL_DATASET_SHA, f"Sai SHA-256 dataset: {actual_ds_sha} != {CANONICAL_DATASET_SHA}"
print(f"[+] Dataset Stage 1 verified (read-only): {DRIVE_DATASET_PATH.name}")

# Exact binding for Stage 2 code archive
code_archive = DRIVE_INPUT_DIR / CANONICAL_CODE_ARCHIVE_NAME
if not code_archive.is_file():
    raise FileNotFoundError(f"Thiếu file code archive canonical tại: {code_archive}")

# Streaming hash and byte size check on Drive before staging
assert code_archive.stat().st_size == CANONICAL_CODE_ARCHIVE_BYTES, (
    f"Sai kích thước code archive trên Drive: {code_archive.stat().st_size} != {CANONICAL_CODE_ARCHIVE_BYTES}"
)
drive_code_sha = sha256_file(code_archive)
assert drive_code_sha == CANONICAL_CODE_ARCHIVE_SHA256, (
    f"Sai SHA-256 code archive trên Drive: {drive_code_sha} != {CANONICAL_CODE_ARCHIVE_SHA256}"
)
print(f"[+] Canonical code archive verified on Drive: {code_archive.name}")

operator_script = DRIVE_INPUT_DIR / "phase_4c2_execute_all.sh"
if not operator_script.is_file():
    raise FileNotFoundError(f"Thiếu script operator: {operator_script}")

# Verify operator size and SHA-256 on Drive
assert operator_script.stat().st_size == CANONICAL_OPERATOR_BYTES, (
    f"Sai kích thước operator: {operator_script.stat().st_size} != {CANONICAL_OPERATOR_BYTES}"
)
actual_op_sha = sha256_file(operator_script)
assert actual_op_sha == CANONICAL_OPERATOR_SHA, (
    f"Sai SHA-256 operator: {actual_op_sha} != {CANONICAL_OPERATOR_SHA}"
)
print(f"[+] Operator script verified on Drive: {operator_script.name} ({actual_op_sha[:8]}...)")

# Stage input artifacts locally via .part and os.replace (atomic restaging)
LOCAL_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)
for src, expected_bytes, expected_sha in [
    (code_archive, CANONICAL_CODE_ARCHIVE_BYTES, CANONICAL_CODE_ARCHIVE_SHA256),
    (operator_script, CANONICAL_OPERATOR_BYTES, CANONICAL_OPERATOR_SHA),
]:
    dst = LOCAL_TRANSFER_DIR / src.name
    part_dst = LOCAL_TRANSFER_DIR / f"{src.name}.part"
    needs_copy = True
    if dst.exists():
        if dst.stat().st_size == expected_bytes and sha256_file(dst) == expected_sha:
            needs_copy = False

    if needs_copy:
        if part_dst.exists():
            part_dst.unlink()
        shutil.copy2(src, part_dst)
        assert part_dst.stat().st_size == expected_bytes, f"Lỗi copy size {src.name}"
        assert sha256_file(part_dst) == expected_sha, f"Lỗi copy SHA {src.name}"
        os.replace(part_dst, dst)
        print(f"[*] Staged {src.name} to {dst}")
    else:
        print(f"[+] Reusing verified local copy: {dst.name}")

(LOCAL_TRANSFER_DIR / operator_script.name).chmod(0o755)

# Setup persistent output directory only after input validation passes
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("[READY] Preflight và staging hoàn tất")


In [ ]:
op_script = LOCAL_TRANSFER_DIR / "phase_4c2_execute_all.sh"
exact_execution_dir = DRIVE_OUTPUT_DIR / f"execution_{CANONICAL_EXECUTION_SHORT_SHA}"

cmd = ["bash", str(op_script)]
if EXECUTE:
    print("[RUN] Bắt đầu 15 Stage 2 partial fine-tuning runs (--execute)")
    cmd.append("--execute")
else:
    print("[PREFLIGHT] Chạy kiểm tra tiền kiểm an toàn (--preflight-only)")
    cmd.append("--preflight-only")

env = dict(os.environ)
env["DATASET_ARCHIVE"] = str(DRIVE_DATASET_PATH)
env["CODE_ARCHIVE"] = str(LOCAL_TRANSFER_DIR / CANONICAL_CODE_ARCHIVE_NAME)
env["INPUT_DIR"] = str(LOCAL_TRANSFER_DIR)

try:
    subprocess.run(cmd, env=env, check=True)
except subprocess.CalledProcessError as e:
    print(f"\n[-] FATAL: Operator thất bại với mã lỗi {e.returncode}")
    failure_report = exact_execution_dir / "OPERATOR_FAILURE.json"
    console_log = exact_execution_dir / "logs" / "operator_console.log"
    if failure_report.is_file():
        print("=" * 60)
        print("[*] NỘI DUNG OPERATOR_FAILURE.json:")
        print("=" * 60)
        print(failure_report.read_text(encoding="utf-8"))
    if console_log.is_file():
        log_lines = console_log.read_text(encoding="utf-8", errors="replace").splitlines()
        tail_lines = log_lines[-100:]
        print("=" * 60)
        print(f"[*] 100 DÒNG CUỐI CONSOLE LOG ({len(tail_lines)}/{len(log_lines)}):")
        print("=" * 60)
        for line in tail_lines:
            print(line)
    raise RuntimeError(f"Operator execution failed with exit code {e.returncode}") from e


In [ ]:
REQUIRED_ARCHIVES = [
    "execution_9ee7fdb_run_receipts_metrics.tar.gz",
    "execution_9ee7fdb_run_predictions.tar.gz",
    "execution_9ee7fdb_run_histories.tar.gz",
    "execution_9ee7fdb_run_checkpoints.tar.gz",
    "execution_9ee7fdb_environment_checksums.tar.gz",
]

if not EXECUTE:
    print("[INFO] Preflight-only hoàn tất thành công. Đổi EXECUTE = True và chọn Run all để huấn luyện 15 runs Stage 2.")
else:
    final_exec_dir = DRIVE_OUTPUT_DIR / f"execution_{CANONICAL_EXECUTION_SHORT_SHA}"
    assert final_exec_dir.is_dir(), f"Không tìm thấy thư mục execution canonical tại: {final_exec_dir}"

    status_file = final_exec_dir / "OPERATOR_STATUS.json"
    assert status_file.is_file(), f"Thiếu file OPERATOR_STATUS.json tại: {status_file}"
    with open(status_file, "r", encoding="utf-8") as f:
        op_status = json.load(f)
    assert op_status.get("status") == "completed", f"Operator status không phải completed: {op_status.get('status')}"
    assert op_status.get("mode") == "execute", f"Operator mode không phải execute: {op_status.get('mode')}"
    assert op_status.get("stage2_invocations") == 1, f"stage2_invocations sai: {op_status.get('stage2_invocations')}"
    assert op_status.get("training_runs_completed") == 15, f"training_runs_completed sai: {op_status.get('training_runs_completed')}"

    download_dir = final_exec_dir / "download"
    assert download_dir.is_dir(), f"Thiếu thư mục download: {download_dir}"

    for name in REQUIRED_ARCHIVES:
        archive_path = download_dir / name
        sidecar_path = download_dir / f"{name}.sha256"
        assert archive_path.is_file(), f"Thiếu archive: {archive_path}"
        assert sidecar_path.is_file(), f"Thiếu sidecar: {sidecar_path}"
        assert archive_path.stat().st_size > 0, f"Archive rỗng: {archive_path}"
        expected_hash = sidecar_path.read_text(encoding="utf-8").strip().split()[0].lower()
        actual_hash = sha256_file(archive_path)
        assert actual_hash == expected_hash, f"Sai SHA-256 archive {name}: {actual_hash} != {expected_hash}"

    sample_sizes = (50, 100, 250)
    seeds = (42, 1337, 2025, 3407, 9001)

    run_dirs = [p for p in final_exec_dir.glob("n*_seed_*") if p.is_dir()]
    assert len(run_dirs) == 15, f"Kỳ vọng đúng 15 run directories, tìm thấy {len(run_dirs)}"

    completed_runs = 0
    for size in sample_sizes:
        for seed in seeds:
            run_dir = final_exec_dir / f"n{size}_seed_{seed}"
            assert run_dir.is_dir(), f"Thiếu thư mục run: {run_dir.name}"
            receipt_path = run_dir / "run_receipt.json"
            assert receipt_path.is_file(), f"Thiếu receipt: {receipt_path}"
            with open(receipt_path, "r", encoding="utf-8") as f:
                receipt = json.load(f)
            assert receipt.get("status") == "completed", f"Run {run_dir.name} chưa completed: {receipt.get('status')}"
            assert receipt.get("stage") == "partial_finetune", f"Sai stage: {receipt.get('stage')}"
            assert receipt.get("sample_size") == size, f"Sai sample size: {receipt.get('sample_size')} != {size}"
            assert receipt.get("seed") == seed, f"Sai seed: {receipt.get('seed')} != {seed}"
            assert receipt.get("treatment_designation") == "pre-registered partial fine-tuning protocol", f"Sai treatment_designation"
            assert receipt.get("trainable_parameters_count") == 204674, f"Sai trainable_parameters_count"
            assert receipt.get("validation_source_count") == 91, f"Sai validation_source_count"
            assert receipt.get("locked_test_access") == 0, f"locked_test_access khác 0 trong {run_dir.name}"
            assert receipt.get("stage1_output_writes") == 0, f"stage1_output_writes khác 0 trong {run_dir.name}"
            completed_runs += 1

    assert completed_runs == 15, f"Kỳ vọng đúng 15 completed runs, đếm được: {completed_runs}"
    print(f"[PASS] Hoàn tất và xác thực đầy đủ 15/15 runs Stage 2 trong {final_exec_dir.name}")
